In [1]:
import numpy as np

def max_pool2d(input, kernel_size, stride=1, padding=0):
    """
    手动实现二维最大池化前向传播
    input: 输入特征图，形状 (H, W) 或 (C, H, W) 或 (N, C, H, W)
    kernel_size: 池化窗口大小 (h, w) 或 int
    stride: 步幅 (s_h, s_w) 或 int
    padding: 填充数 (p_h, p_w) 或 int
    """
    # 统一处理参数格式
    if isinstance(kernel_size, int):
        kernel_size = (kernel_size, kernel_size)
    if isinstance(stride, int):
        stride = (stride, stride)
    if isinstance(padding, int):
        padding = (padding, padding)

    # 获取输入形状并处理填充
    original_ndim = input.ndim
    if original_ndim == 2:
        input = input[np.newaxis, np.newaxis, :, :]  # (1,1,H,W)
    elif original_ndim == 3:
        input = input[np.newaxis, :, :, :]           # (1,C,H,W)
    # 现在 input 形状为 (N, C, H, W)

    N, C, H, W = input.shape
    kh, kw = kernel_size
    sh, sw = stride
    ph, pw = padding

    # 填充
    pad_width = ((0,0), (0,0), (ph, ph), (pw, pw))
    input_pad = np.pad(input, pad_width, mode='constant', constant_values=0)

    # 计算输出尺寸
    out_h = (H + 2*ph - kh) // sh + 1
    out_w = (W + 2*pw - kw) // sw + 1
    output = np.zeros((N, C, out_h, out_w))

    # 滑动窗口取最大值
    for i in range(out_h):
        for j in range(out_w):
            h_start = i * sh
            h_end = h_start + kh
            w_start = j * sw
            w_end = w_start + kw
            window = input_pad[:, :, h_start:h_end, w_start:w_end]
            output[:, :, i, j] = np.max(window, axis=(2,3))

    # 恢复原始维度
    if original_ndim == 2:
        output = output[0,0]
    elif original_ndim == 3:
        output = output[0]
    return output

# 示例测试
if __name__ == "__main__":
    x = np.random.randn(1, 1, 4, 4)  # 单通道 4x4
    out = max_pool2d(x, kernel_size=2, stride=2, padding=0)
    print("池化输出形状:", out.shape)  # (1,1,2,2)

池化输出形状: (1, 1, 2, 2)


In [2]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.block(x)

# 示例：输入通道3，输出通道16，卷积核3x3，步幅1，填充1
block = NiNBlock(3, 16, kernel_size=3, stride=1, padding=1)
x = torch.randn(1, 3, 32, 32)
out = block(x)
print("NiN块输出形状:", out.shape)  # (1,16,32,32)

NiN块输出形状: torch.Size([1, 16, 32, 32])


In [3]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        # 第一个卷积层
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        # 第二个卷积层
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # 如果输入输出通道数不同或需要调整尺寸，使用1x1卷积
        if use_1x1conv or stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.shortcut = nn.Identity()

        self.relu = nn.ReLU()

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        return self.relu(out)

# 示例
block = Residual(3, 3, use_1x1conv=False)
x = torch.randn(1, 3, 32, 32)
out = block(x)
print("残差块输出形状:", out.shape)  # (1,3,32,32)

残差块输出形状: torch.Size([1, 3, 32, 32])


In [4]:
import torchvision.transforms as transforms

augmentation_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),   # 随机裁剪并缩放至224x224
    transforms.RandomHorizontalFlip(p=0.5),                # 50%概率水平翻转
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),  # 颜色抖动
    transforms.ToTensor()                                   # 转换为张量
])

# 使用示例（需要PIL图像）
# from PIL import Image
# img = Image.open('test.jpg')
# tensor_img = augmentation_pipeline(img)
# print(tensor_img.shape)  # (3, 224, 224)

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.smoothing = smoothing

    def forward(self, pred, target):
        """
        pred: 模型输出 logits，形状 (N, C)
        target: 真实标签，形状 (N,)
        """
        n_classes = pred.size(1)
        # 构造平滑后的标签分布
        with torch.no_grad():
            # 真实类别的概率为 1 - smoothing
            smooth_target = torch.zeros_like(pred).fill_(self.smoothing / (n_classes - 1))
            smooth_target.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
        # 计算交叉熵（等价于对每个样本计算 -sum(q_i * log(p_i))）
        log_prob = F.log_softmax(pred, dim=1)
        loss = - (smooth_target * log_prob).sum(dim=1).mean()
        return loss

# 示例
criterion = LabelSmoothingCrossEntropy(smoothing=0.1)
pred = torch.randn(4, 10)   # 4个样本，10分类
target = torch.tensor([1, 2, 3, 4])
loss = criterion(pred, target)
print("标签平滑损失:", loss.item())

标签平滑损失: 3.557361602783203
